# Assignment 3

In [1]:
import numpy as np
from sklearn.svm import SVC

## Training Data (Table 1)

In [7]:
X_train = np.array([
    [0, 0, 0], # +
    [0, 0, 1], # -
    [0, 1, 0], # -
    [0, 1, 1], # -
    [1, 0, 0], # +
    [1, 0, 1], # -
    [1, 1, 0], # -
    [1, 1, 1], # +
])
y_train = np.array([1, 0, 0, 0, 1, 0, 0, 1])

pos = X_train[y_train == 1]
neg = X_train[y_train == 0]

print(f"Total samples : {len(y_train)}")
print(f"Positive (+)  : {len(pos)}")
print(f"Negative (-)  : {len(neg)}")

Total samples : 8
Positive (+)  : 3
Negative (-)  : 5


## Q1a: Naïve Bayes Probability Estimates

In [3]:
P_pos = len(pos)/len(y_train)
P_neg = len(neg)/len(y_train)

print(f"P(+) = {len(pos)}/{len(y_train)} = {P_pos:.4f}")
print(f"P(-) = {len(neg)}/{len(y_train)} = {P_neg:.4f}")
print()

for feature, name in zip(range(3), ['A', 'B', 'C']):
    for val in [1, 0]:
        p_given_pos = np.sum(pos[:, feature] == val) / len(pos)
        p_given_neg = np.sum(neg[:, feature] == val) / len(neg)
        print(f"P({name}={val}|+) = {np.sum(pos[:, feature] == val)}/{len(pos)} = {p_given_pos:.4f}   "
              f"P({name}={val}|-) = {np.sum(neg[:, feature] == val)}/{len(neg)} = {p_given_neg:.4f}")

P(+) = 3/8 = 0.3750
P(-) = 5/8 = 0.6250

P(A=1|+) = 2/3 = 0.6667   P(A=1|-) = 2/5 = 0.4000
P(A=0|+) = 1/3 = 0.3333   P(A=0|-) = 3/5 = 0.6000
P(B=1|+) = 1/3 = 0.3333   P(B=1|-) = 3/5 = 0.6000
P(B=0|+) = 2/3 = 0.6667   P(B=0|-) = 2/5 = 0.4000
P(C=1|+) = 1/3 = 0.3333   P(C=1|-) = 3/5 = 0.6000
P(C=0|+) = 2/3 = 0.6667   P(C=0|-) = 2/5 = 0.4000


## Q1b: Naïve Bayes Prediction for (A=1, B=0, C=0)

In [8]:
test_q1 = [1, 0, 0]

score_pos = P_pos
score_neg = P_neg

print("P(+|x) = P(+) × P(A=1|+) × P(B=0|+) × P(C=0|+)")
print(f"       = {P_pos:.4f}", end="")
for feature, val in enumerate(test_q1):
    p = np.sum(pos[:, feature] == val) / len(pos)
    score_pos *= p
    print(f" × {p:.4f}", end="")
print(f" = {score_pos:.6f}")

print()
print("P(-|x) = P(-) × P(A=1|-) × P(B=0|-) × P(C=0|-)")
print(f"       = {P_neg:.4f}", end="")
for feature, val in enumerate(test_q1):
    p = np.sum(neg[:, feature] == val) / len(neg)
    score_neg *= p
    print(f" × {p:.4f}", end="")
print(f" = {score_neg:.6f}")

print()
pred_q1 = '+' if score_pos > score_neg else '-'
print(f"Since {score_pos:.6f} {'>' if score_pos > score_neg else '<'} {score_neg:.6f}")
print(f"Predicted class: {pred_q1}")

P(+|x) ∝ P(+) × P(A=1|+) × P(B=0|+) × P(C=0|+)
       = 0.3750 × 0.6667 × 0.6667 × 0.6667 = 0.111111

P(-|x) ∝ P(-) × P(A=1|-) × P(B=0|-) × P(C=0|-)
       = 0.6250 × 0.4000 × 0.4000 × 0.4000 = 0.040000

Since 0.111111 > 0.040000
Predicted class: +


## Q2: SVM Prediction for (A=0, B=1, C=0)

In [5]:
model = SVC(kernel='rbf', C=1.0, gamma='scale')
model.fit(X_train, y_train)

test_q2 = np.array([[0, 1, 0]])
prediction = model.predict(test_q2)
pred_q2 = '+' if prediction[0] == 1 else '-'

print(f"SVM (kernel=rbf, C=1.0, gamma='scale')")
print(f"Prediction for (A=0, B=1, C=0): class {pred_q2}")

SVM (kernel=rbf, C=1.0, gamma='scale')
Prediction for (A=0, B=1, C=0): class -


## Q3: Perceptron Weights for NOR

The perceptron fires when W_1 X_1 + W_2 X_2 + b >= 0.

NOR outputs 1 only when both inputs are 0, so we need:
- (0,0): b >= 0 -> willfire
- (0,1): W_2 + b < 0 -> doesn't fire
- (1,0): W_1 + b < 0 -> doesn't fire
- (1,1): W_1 + W_2 + b < 0 -> doesn't fire

Choosing **W1 = −1, W2 = −1, b = 0.5** satisfies all constraints.

In [13]:
W1 = -1
W2 = -1
b = 0.5

print(f"Weights: W1={W1}, W2={W2}, bias b={b}")
print()
print(f"{'X1':>4} {'X2':>4} {'Expected NOR':>14} {'Activation':>12} {'Output':>8} {'Match':>6}")

nor_table = [(0, 0, 1), (0, 1, 0), (1, 0, 0), (1, 1, 0)]
all_correct = True
for x1, x2, expected in nor_table:
    activation = W1 * x1 + W2 * x2 + b
    output = 1 if activation >= 0 else 0
    match = "1" if output == expected else "0" #1 for true, 0 for false
    if output != expected:
        all_correct = False
    print(f"{x1:>4} {x2:>4} {expected:>14} {activation:>12.1f} {output:>8}  {match}")

Weights: W1=-1, W2=-1, bias b=0.5

  X1   X2   Expected NOR   Activation   Output  Match
   0    0              1          0.5        1  1
   0    1              0         -0.5        0  1
   1    0              0         -0.5        0  1
   1    1              0         -1.5        0  1
